In [ ]:
!pip install xgboost lightgbm --quiet
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib

In [ ]:
file_path = "/kaggle/input/findata-auth/coherent_access_auth_dataset_with_risk.csv"
df = pd.read_csv(file_path)
print(df.head(10)) 
print(df.columns)

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder

# Assuming df is the new dataset
# Handle missing values if any
df = df.dropna()


# Encoding categorical features using LabelEncoder
label_cols = ['DeviceType', 'OS_BrowserInfo', 'MFAStatus', 'APIAccess', 'PrivilegedAccess']
encoder = LabelEncoder()

for col in label_cols:
    df[col] = encoder.fit_transform(df[col])

# Extract features from LoginTimestamp
df['LoginTimestamp'] = pd.to_datetime(df['LoginTimestamp'])
df['LoginHour'] = df['LoginTimestamp'].dt.hour
df['LoginDay'] = df['LoginTimestamp'].dt.day
df['LoginWeekday'] = df['LoginTimestamp'].dt.weekday
df['LoginMonth'] = df['LoginTimestamp'].dt.month
df['LoginYear'] = df['LoginTimestamp'].dt.year

# Scaling the continuous numerical columns
scaler = MinMaxScaler()
# Adjusted to scale only the columns that exist in your dataset
continuous_cols = ['SessionDuration']  
df[continuous_cols] = scaler.fit_transform(df[continuous_cols])

# Drop columns that are not needed anymore (except UserID and Geolocation)
df.drop(columns=['LogID', 'UserID', 'LoginTimestamp', 'IPAddress', 'Geolocation'], inplace=True)

# Checking the preprocessed data
print(df.head(20))

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['RiskLevel'])
y = df['RiskLevel']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training Set: X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"Testing Set: X_test: {X_test.shape}, y_test: {y_test.shape}")

In [ ]:
# base_models = [
#     ('linear', LinearRegression()),
#     ('random_forest', RandomForestRegressor(n_estimators=200, random_state=42)),
#     ('xgboost', XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42)),
#     ('lightgbm', LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42))
# ]

# # Stacking model using GradientBoostingRegressor as final estimator
# stacking_model = StackingRegressor(estimators=base_models, final_estimator=GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42))

# # 8. Train the Stacking Model
# print("\nTraining the Stacking Model...")
# stacking_model.fit(X_train, y_train)

# # 9. Predict on the Test Data
# y_pred = stacking_model.predict(X_test)

# # 10. Evaluate Model Performance
# mse = mean_squared_error(y_test, y_pred)
# r2 = r2_score(y_test, y_pred)

# print(f"\n Model Performance:")
# print(f"MSE = {mse:.4f}")
# print(f"R² Score = {r2:.4f}")

In [ ]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor

# Define parameter grid for tuning
param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 6, 9],
    'subsample': [0.8, 1.0]
}

# Initialize XGBoost model
xgboost_model = XGBRegressor(random_state=42)

# Grid search for hyperparameter tuning
grid_search = GridSearchCV(estimator=xgboost_model, param_grid=param_grid, scoring='neg_mean_squared_error', cv=3)
grid_search.fit(X_train, y_train)

# Best model and predictions
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Best XGBoost Model Performance:")
print(f"MSE = {mse:.4f}")
print(f"R² Score = {r2:.4f}")

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import numpy as np

# Example of raw input data (replace with your actual input)
raw_input_data = {
    'LogID': ['fdaf345b-74f3-40e1-ad9b-99fe38510770'],
    'UserID': ['U57248'],
    'LoginTimestamp': ['2025-02-01 04:31:35'],
    'FailedLoginAttempts': [0],
    'DeviceType': ['PC'],
    'OS_BrowserInfo': ['iOS-App'],
    'IPAddress': ['36.161.195.1'],
    'Geolocation': ['Danielleville'],
    'MFAStatus': ['Enabled'],
    'SessionDuration': [29.34],
    'APIAccess': ['Yes'],
    'PrivilegedAccess': ['Guest']
}

# Convert to DataFrame
raw_input_df = pd.DataFrame(raw_input_data)

# Preprocess the raw input (same steps as for training data)
# 1. Encoding categorical features
label_cols = ['DeviceType', 'OS_BrowserInfo', 'MFAStatus', 'APIAccess', 'PrivilegedAccess']
encoder = LabelEncoder()

for col in label_cols:
    raw_input_df[col] = encoder.fit_transform(raw_input_df[col])

# 2. Extract features from LoginTimestamp
raw_input_df['LoginTimestamp'] = pd.to_datetime(raw_input_df['LoginTimestamp'])
raw_input_df['LoginHour'] = raw_input_df['LoginTimestamp'].dt.hour
raw_input_df['LoginDay'] = raw_input_df['LoginTimestamp'].dt.day
raw_input_df['LoginWeekday'] = raw_input_df['LoginTimestamp'].dt.weekday
raw_input_df['LoginMonth'] = raw_input_df['LoginTimestamp'].dt.month
raw_input_df['LoginYear'] = raw_input_df['LoginTimestamp'].dt.year

# 3. Scaling continuous features (SessionDuration)
scaler = MinMaxScaler()
raw_input_df['SessionDuration'] = scaler.fit_transform(raw_input_df[['SessionDuration']])

# 4. Drop unnecessary columns (LogID, UserID, IPAddress, Geolocation)
raw_input_df.drop(columns=['LogID', 'UserID', 'IPAddress', 'Geolocation'], inplace=True)

# 5. Make sure the columns match the training data columns
raw_input_df = raw_input_df[X_train.columns]

# Make the prediction using the best model from GridSearchCV
y_pred = best_model.predict(raw_input_df)

# Print the predicted RiskLevel
print(f"Predicted RiskLevel: {y_pred[0]:.4f}")

In [ ]:
import matplotlib.pyplot as plt

# Access feature importances
importances = best_model.feature_importances_

# Plot the feature importances
plt.figure(figsize=(10, 6))
plt.barh(range(len(importances)), importances)
plt.yticks(range(len(importances)), X_train.columns)  # Replace X_train.columns with your actual feature names
plt.xlabel('Feature Importance')
plt.title('Feature Importances for XGBRegressor')
plt.show()

In [ ]:
import shap

# Create a SHAP explainer
explainer = shap.TreeExplainer(best_model)

# Compute SHAP values
shap_values = explainer.shap_values(X_test)

# Plot the SHAP summary plot
shap.summary_plot(shap_values, X_test)

In [ ]:
from sklearn.linear_model import LinearRegression

# Fit a linear regression model (as an example)
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

# Get the coefficients and intercept
coefficients = linear_model.coef_
intercept = linear_model.intercept_

# Print the equation
equation = "y = " + str(intercept)
for i, coef in enumerate(coefficients):
    equation += f" + ({coef})*X{i}"
    
print(f"Equation: {equation}")